# Flood Change Detection with Sentinel-1 Data

openEO can be used for a wide range of Earth Observation (EO) applications, from simple to advanced workflows. In this notebook, we will demonstrate a simple workflow to use openEO for **flood change detection using Sentinel-1 data**.

The workflow compares radar imagery before and after a flood event to identify newly inundated areas.


Let us start by connecting to the Copernicus Data Space Ecosystem (CDSE) backend, which provides access to Sentinel-1 data and processing capabilities.

In [1]:
import openeo

connection = openeo.connect("https://openeo.dataspace.copernicus.eu/").authenticate_oidc()
connection.capabilities()

Authenticated using refresh token.


[Storm Boris](http://nl.wikipedia.org/wiki/Boris_(cycloon)) brought severe flooding to Central Europe in September 2024. It caused significant damage in the Czech Republic, Poland, and Austria. In this case study, we will explore for a specific location in the Czech Republic. 

In [3]:
spatial_extent = {"west": 17.05,"south": 50.07,"east": 17.31,"north": 50.25}

# plot spatial extent
import folium
map = folium.Map(
    location=[(spatial_extent["south"] + spatial_extent["north"]) / 2, (spatial_extent["west"] + spatial_extent["east"]) / 2], 
    zoom_start=13
    )
folium.Rectangle(
    bounds=[[spatial_extent["south"], spatial_extent["west"]], [spatial_extent["north"], spatial_extent["east"]]],
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.2
).add_to(map)
map


Next, let us load the Sentinel-1 GRD data from the CDSE backend for our area of interest and time range, and load the relevant bands for flood detection (VV and VH polarizations).

Next, using the inbuilt processes in openEO such as `sar_backscatter`, we can easily compute the backscatter coefficients needed for flood detection without having to write complex code for data handling and processing.

In [4]:
before_cube = connection.load_collection(        
                            "SENTINEL1_GRD",        
                            spatial_extent=spatial_extent,        
                            temporal_extent=["2024-08-25", "2024-09-09"],        
                            bands=["VV"],        
                            properties={"sat:orbit_state": lambda x: x == "descending"})    
# applying radiometric corrections
before_cube = before_cube.sar_backscatter(coefficient="sigma0-ellipsoid")

Using the `median_time` process of openEO, we can compute the median backscatter value for each pixel across the time series, which helps to reduce noise and improve the accuracy of flood detection.

In [5]:
# temporal reduction to get a single image for the before period
before_median = before_cube.median_time()    

Similarly, the median backscatter values for post-flood is calculated as shown below.

In [6]:
after_cube = connection.load_collection(        
                            "SENTINEL1_GRD",        
                            spatial_extent=spatial_extent,        
                            temporal_extent=["2024-09-11", "2024-09-25"],        
                            bands=["VV"],        
                            properties={"sat:orbit_state": lambda x: x == "descending"}) 
after_cube = after_cube.sar_backscatter(coefficient="sigma0-ellipsoid")    
after_median = after_cube.median_time()    


To identify water bodies in the radar imagery, we can apply a backscatter threshold to the computed median backscatter logarithmic values. 

```
0  dB -> Strong reflector (metal, corner reflector)
     ║
-5  dB -> Urban areas, concrete
     ║
-10 dB -> Forest, rough terrain
     ║
-15 dB -> WATER THRESHOLD!
     ║
-20 dB -> Calm water (lakes, smooth areas)
     ║
-25 dB -> Very smooth water
```

The backscatter coefficient (in dB) for water bodies is typically lower than that of land surfaces due to the smoothness of water, which results in less radar signal being reflected back to the sensor.

In [7]:
threshold = -15   
before_water = before_median.apply(
                        lambda x: openeo.processes.if_(
                            10 * openeo.processes.log(x, base=10) < threshold, 
                            1,0)
                            )
before_water_result = before_water.save_result(format="GTiff", options={"filename_prefix": "before_water"})

after_water = after_median.apply(
                        lambda x: openeo.processes.if_(
                            10 * openeo.processes.log(x, base=10) < threshold, 
                            1,0)
                            )
after_water_result = after_water.save_result(format="GTiff", options={"filename_prefix": "after_water"})

Furthermore, to efficiently compare the pre and post-flood outputs, let us save intermediate results in a single openEO workflow. This avoids re-running the same processes multiple times. For more information on saving intermediate results in the workflow, please refer to the [openEO documentation](https://open-eo.github.io/openeo-python-client/api.html#openeo.rest.multiresult.MultiResult).

Next, we can simply compare the binary water masks for pre and post-flood periods to identify newly flooded areas. This can be done using a simple subtraction operation, where pixels that were non-water before but are water after the flood will be highlighted as newly inundated areas.

In [8]:
flood_change = after_water - before_water    
flood_change_result = flood_change.save_result(format="GTiff", options={"filename_prefix": "change_detection"})
flood_change_result

Please note that while this looks like an actual calculation, real data processing still needs to be done. At this point, the **flood_change_result** object is just an abstract representation of our algorithm under construction. The mathematical operators we used here are syntactic sugar for compactly expressing this part of the algorithm.

In [9]:
saving_multiple_result = openeo.MultiResult(
        [flood_change_result, before_water_result, after_water_result]
    )

Finally, to trigger an actual execution (on the backend), we have to explicitly send the above representation to the backend. We can do this either synchronously(simple download) or using the batch-job-based method. 

Most of the simple, basic openEO usage examples show synchronous downloading of results. Synchronous downloads are handy for quick experimentation on small data cubes.

This only works properly if the processing doesn’t take too long and is focused on a smaller area of interest. However, you have to use batch jobs for the heavier work (larger regions of interest, larger time series, more intensive processing).

For more information on using batch-job in openEO, visit [here](https://open-eo.github.io/openeo-python-client/batch_jobs.html).

In [10]:
openeo_job = saving_multiple_result.create_job(title="Flood Change Detection", description="Detecting flood changes using Sentinel-1 data")
openeo_job.start_and_wait()

0:00:00 Job 'j-260409144508448984031b1951605ff3': send 'start'
0:00:16 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:00:21 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:00:28 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:00:36 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:00:46 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:00:58 Job 'j-260409144508448984031b1951605ff3': created (progress 0%)
0:01:14 Job 'j-260409144508448984031b1951605ff3': running (progress N/A)
0:01:33 Job 'j-260409144508448984031b1951605ff3': running (progress N/A)
0:01:58 Job 'j-260409144508448984031b1951605ff3': running (progress N/A)
0:02:28 Job 'j-260409144508448984031b1951605ff3': running (progress N/A)
0:03:05 Job 'j-260409144508448984031b1951605ff3': running (progress N/A)
0:03:53 Job 'j-260409144508448984031b1951605ff3': finished (progress 100%)


<BatchJob job_id='j-260409144508448984031b1951605ff3'>

#### What Happens When You Submit
1. Your Python code → Process graph (JSON) is sent to backend
2. Job queued
3. Backend optimizes execution plan
4. Executes on distributed infrastructure
5. Results stored temporarily
6. You download results

In [11]:
results = openeo_job.get_results()
results.download_files("my_flood_change_results/")

[WindowsPath('my_flood_change_results/openEO.tif'),
 WindowsPath('my_flood_change_results/job-results.json')]

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

tif_file = Path("openEO.tif")

with rasterio.open(tif_file) as src:
    data = src.read()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    band_names = ["Before Water", "After Water", "Flood Change"]
    
    for idx, (ax, band) in enumerate(zip(axes, data)):
        im = ax.imshow(band, cmap='Blues')
        ax.set_title(band_names[idx])
        ax.set_xlabel('Pixels (X)')
        ax.set_ylabel('Pixels (Y)')
        plt.colorbar(im, ax=ax, label='Value')
    
    plt.tight_layout()
    plt.savefig('flood_detection_results.png', dpi=100, bbox_inches='tight')
    plt.show()

In the above workflow, we have generated three key outputs:
1. **Before Water Map** (binary mask of water bodies before the flood)
2. **After Water Map** (binary mask of water bodies after the flood)
3. **Flood Change Map** (binary mask of newly flooded areas)

These generated maps can be used for further analysis, visualization, or integration into GIS software. They could serve as important inputs for emergency response teams, urban planners, and environmental scientists to assess flood impacts and plan mitigation strategies within few hours after the event, without needing to download and process large datasets locally.

